<a href="https://colab.research.google.com/github/anupbagale/python-for-ai-ml/blob/main/03-oop-for-ml/01_lesson.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 3: Object-Oriented Programming for ML

Every major ML library is built out of classes. In scikit-learn you call `scaler.fit(X)` and then `scaler.transform(X)`. In PyTorch you write `class MyModel(nn.Module)`, call `model(x)`, then `loss.backward()` and `optimizer.step()`. To use these libraries well, and especially to read their code and extend them, you need to be comfortable with classes.

This module teaches object-oriented programming through the patterns those libraries actually use. By the end you'll have written your own versions of a scikit-learn scaler, a PyTorch `Dataset` and `DataLoader`, a baseline model, an ensemble, and a typed training config.

**Contents**

1. Why classes?
2. Classes, instances and `self`
3. Class attributes vs instance attributes
4. Dunder methods: making your objects behave like built-ins
5. Properties
6. Inheritance and abstract base classes
7. Composition
8. Dataclasses
9. Class methods and static methods
10. Putting it together: `Dataset` and `DataLoader` from scratch

## 1. Why classes?

In the Module 1 mini project, we scaled features with two functions:

```python
mins, maxs = fit_minmax(train)
train_scaled = transform(train, mins, maxs)
test_scaled = transform(test, mins, maxs)
```

It worked, but notice that `mins` and `maxs` have to be carried around and passed into `transform` every time. They belong together, and they belong with the code that uses them. Now imagine a pipeline with five preprocessing steps, each with its own learned numbers. You'd be juggling a dozen loose variables, and it would be easy to pass the wrong ones to the wrong function.

A class bundles **data** (the learned numbers) and **behaviour** (the functions that use them) into one object. That's the whole idea. The scikit-learn version of the code above looks like this:

```python
scaler = MinMaxScaler()
scaler.fit(train)
train_scaled = scaler.transform(train)
test_scaled = scaler.transform(test)
```

The scaler remembers what it learned. You can't accidentally pass it the wrong numbers, because you don't pass it numbers at all.

Classes aren't always the answer. A function that takes inputs and returns outputs, with nothing to remember in between, should stay a function. Classes earn their place when there's **state**: things that are learned, accumulated or configured once and used many times.

## 2. Classes, instances and self

Let's build a standard scaler, which shifts values to have mean 0 and standard deviation 1. To keep the focus on the class itself, it works on a plain list of numbers.

In [1]:
class StandardScaler:
    """Scale values to mean 0 and standard deviation 1."""

    def __init__(self):
        self.mean_ = None
        self.std_ = None

    def fit(self, values):
        n = len(values)
        self.mean_ = sum(values) / n
        variance = sum((v - self.mean_) ** 2 for v in values) / n
        self.std_ = variance ** 0.5
        if self.std_ == 0:
            self.std_ = 1.0        # constant input: avoid dividing by zero
        return self

    def transform(self, values):
        if self.mean_ is None:
            raise RuntimeError("call fit() before transform()")
        return [(v - self.mean_) / self.std_ for v in values]

Let's go through the pieces.

- `class StandardScaler:` defines a new type. By convention, class names use `CapitalizedWords`.
- `__init__` runs automatically when you create an object. It sets up the starting state.
- `self` is the object itself. Every method receives it as the first argument, and `self.mean_ = ...` stores a value on that particular object. Values stored like this are called **attributes**.
- `fit` and `transform` are **methods**: functions that belong to the class and can read and change `self`.

Now let's use it:

In [2]:
train = [10.0, 12.0, 14.0, 16.0, 18.0]
test = [11.0, 20.0]

scaler = StandardScaler()       # create an instance: this calls __init__
scaler.fit(train)               # learns mean_ and std_ from train

print(scaler.mean_, scaler.std_)
print(scaler.transform(train))
print(scaler.transform(test))   # uses the numbers learned from train

14.0 2.8284271247461903
[-1.414213562373095, -0.7071067811865475, 0.0, 0.7071067811865475, 1.414213562373095]
[-1.0606601717798212, 2.1213203435596424]


`StandardScaler` is the **class**, like a blueprint. `scaler` is an **instance**, one object built from that blueprint. You can make as many instances as you like, and each one has its own attributes:

In [3]:
heights = StandardScaler().fit([150, 160, 170, 180])
weights = StandardScaler().fit([50, 65, 80])

print(heights.mean_, weights.mean_)

165.0 65.0


### What self really is

When you write `scaler.fit(train)`, Python actually calls `StandardScaler.fit(scaler, train)`. The object before the dot is passed in as the first argument. That's all `self` is. The name `self` is just a convention, but it's a universal one, so stick to it.

In [4]:
s = StandardScaler()
StandardScaler.fit(s, [1.0, 2.0, 3.0])   # exactly what s.fit([1.0, 2.0, 3.0]) does
print(s.mean_)

2.0


### Two conventions worth copying from scikit-learn

**`fit` returns `self`.** That's why `StandardScaler().fit([150, 160, 170, 180])` above gave us back the scaler, and it lets you chain calls: `StandardScaler().fit(train).transform(test)`.

**Learned attributes end with an underscore**, like `mean_` and `std_`. It tells the reader "this value only exists after `fit` has run". Settings you pass in at creation time don't get the underscore.

### Private by convention

Python doesn't have truly private attributes. Instead, a leading underscore, as in `self._cache`, means "internal, don't touch this from outside the class". Nothing stops you, but it's a clear signal, and you'll see it all over library code.

## 3. Class attributes vs instance attributes

Attributes set on `self` belong to one instance. Attributes defined directly in the class body are shared by **all** instances:

In [5]:
class Model:
    framework = "pure-python"        # class attribute: shared

    def __init__(self, name):
        self.name = name             # instance attribute: one per object

a = Model("baseline")
b = Model("improved")
print(a.name, b.name)
print(a.framework, b.framework, Model.framework)

baseline improved
pure-python pure-python pure-python


Class attributes are fine for constants. But a mutable class attribute is the same trap as a mutable default argument from Module 1:

In [6]:
class Experiment:
    history = []                     # bug: one list shared by every experiment

    def log(self, loss):
        self.history.append(loss)

exp1 = Experiment()
exp2 = Experiment()
exp1.log(0.9)
exp2.log(0.5)

print(exp1.history)                  # contains exp2's loss too

[0.9, 0.5]


The fix is to create the list in `__init__`, so every instance gets its own:

In [7]:
class Experiment:
    def __init__(self):
        self.history = []

    def log(self, loss):
        self.history.append(loss)

exp1 = Experiment()
exp2 = Experiment()
exp1.log(0.9)
exp2.log(0.5)
print(exp1.history, exp2.history)

[0.9] [0.5]


**Rule:** anything mutable that belongs to one object goes in `__init__`.

## 4. Dunder methods

Methods with double underscores on both sides, like `__init__`, are called "dunder" (double underscore) methods, or magic methods. Python calls them for you when you use built-in syntax. Implement the right ones and your objects work with `print`, `len`, `[]`, `for`, `in`, `==`, `+` and even function-call syntax.

### `__repr__`: a readable printout

By default, printing an object isn't very helpful:

In [8]:
print(StandardScaler())

`__repr__` returns the string Python shows for your object. A good one looks like the code you'd write to create the object:

In [9]:
class LinearModel:
    def __init__(self, weight, bias):
        self.weight = weight
        self.bias = bias

    def __repr__(self):
        return f"LinearModel(weight={self.weight}, bias={self.bias})"

model = LinearModel(2.0, 1.0)
print(model)
print([model, LinearModel(0.5, -1.0)])   # lists use __repr__ for their items

LinearModel(weight=2.0, bias=1.0)
[LinearModel(weight=2.0, bias=1.0), LinearModel(weight=0.5, bias=-1.0)]


This pays off every time you debug. When you print a list of models or configs, you see what's in them.

There's also `__str__`, which `print` prefers if it exists, meant for a friendlier, user-facing version. If you only write one, write `__repr__`.

### `__call__`: objects you can call like functions

In PyTorch you run a model with `model(x)`, not `model.forward(x)`. That works because `nn.Module` defines `__call__`. We can do the same:

In [10]:
class LinearModel:
    def __init__(self, weight, bias):
        self.weight = weight
        self.bias = bias

    def __repr__(self):
        return f"LinearModel(weight={self.weight}, bias={self.bias})"

    def __call__(self, x):
        return self.weight * x + self.bias

model = LinearModel(2.0, 1.0)
print(model(3.0))
print([model(x) for x in [0, 1, 2]])

7.0
[1.0, 3.0, 5.0]


### `__len__` and `__getitem__`: behaving like a sequence

This pair is how PyTorch datasets work. A `Dataset` needs exactly two things: `__len__`, so `len(dataset)` works, and `__getitem__`, so `dataset[i]` returns example number `i`.

In [11]:
class TextDataset:
    def __init__(self, texts, labels):
        if len(texts) != len(labels):
            raise ValueError("texts and labels must have the same length")
        self.texts = texts
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, i):
        return self.texts[i], self.labels[i]

    def __repr__(self):
        return f"TextDataset({len(self)} examples)"


ds = TextDataset(["great film", "awful plot", "loved it"], [1, 0, 1])
print(ds)
print(len(ds))
print(ds[0])
print(ds[-1])

TextDataset(3 examples)
3
('great film', 1)
('loved it', 1)


Because we passed `i` straight through to a list, negative indexes work too. And there's a bonus: if a class has `__getitem__`, Python can loop over it, by calling `ds[0]`, `ds[1]`, and so on until it gets an `IndexError`:

In [12]:
for text, label in ds:
    print(label, text)

1 great film
0 awful plot
1 loved it


The important idea is that the dataset doesn't have to hold all its data in memory. `__getitem__` could load an image from disk, or tokenize text on the fly, only when that example is asked for. That's how you train on datasets far bigger than your RAM.

### `__eq__`: comparing objects

By default, `==` on two objects checks whether they're the *same object*, not whether they hold the same values:

In [13]:
print(LinearModel(2.0, 1.0) == LinearModel(2.0, 1.0))

False


To compare by value, define `__eq__`. Return `NotImplemented` for types you don't know how to compare, so Python can try other options instead of guessing:

In [14]:
class LinearModel:
    def __init__(self, weight, bias):
        self.weight = weight
        self.bias = bias

    def __repr__(self):
        return f"LinearModel(weight={self.weight}, bias={self.bias})"

    def __eq__(self, other):
        if not isinstance(other, LinearModel):
            return NotImplemented
        return self.weight == other.weight and self.bias == other.bias

print(LinearModel(2.0, 1.0) == LinearModel(2.0, 1.0))
print(LinearModel(2.0, 1.0) == "something else")

True
False


### Arithmetic operators

`+` calls `__add__`, `*` calls `__mul__`, and so on. Here's a tiny 2D vector:

In [15]:
class Vec2:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __repr__(self):
        return f"Vec2({self.x}, {self.y})"

    def __add__(self, other):
        return Vec2(self.x + other.x, self.y + other.y)

    def __mul__(self, scalar):
        return Vec2(self.x * scalar, self.y * scalar)

    def __rmul__(self, scalar):
        return self * scalar

v = Vec2(1, 2)
w = Vec2(3, 4)
print(v + w)
print(v * 10)
print(10 * v)

Vec2(4, 6)
Vec2(10, 20)
Vec2(10, 20)


Why do we need `__rmul__`? For `v * 10`, Python calls `v.__mul__(10)`. But for `10 * v`, it first tries `int.__mul__(10, v)`, and the integer has no idea what a `Vec2` is. Python then tries the reflected version on the right-hand object: `v.__rmul__(10)`. Without `__rmul__`, `10 * v` raises a `TypeError`.

These operators are how NumPy arrays and PyTorch tensors let you write `a + b` and `2 * x`, and they're the foundation of the mini project, where `+` and `*` will secretly record a graph of every calculation so we can compute gradients automatically.

### The ones you'll use most

| You write | Python calls |
|---|---|
| `repr(obj)`, `print(obj)` | `__repr__` / `__str__` |
| `len(obj)` | `__len__` |
| `obj[i]` | `__getitem__` |
| `for x in obj` | `__iter__` (or `__getitem__`) |
| `x in obj` | `__contains__` |
| `obj == other` | `__eq__` |
| `obj + other`, `obj * other` | `__add__`, `__mul__` |
| `other + obj` (when `other` can't) | `__radd__` |
| `obj(x)` | `__call__` |

## 5. Properties

Sometimes you want something that looks like a plain attribute but runs code when it's read or set. That's what `@property` is for.

A common use is validation. A classification threshold must be between 0 and 1, and we'd rather find out immediately if someone sets it to 50 by mistake:

In [16]:
class ThresholdClassifier:
    def __init__(self, threshold=0.5):
        self.threshold = threshold        # goes through the setter below

    @property
    def threshold(self):
        return self._threshold

    @threshold.setter
    def threshold(self, value):
        if not 0 <= value <= 1:
            raise ValueError(f"threshold must be between 0 and 1, got {value}")
        self._threshold = value

    def predict(self, probabilities):
        return [1 if p >= self.threshold else 0 for p in probabilities]


clf = ThresholdClassifier()
print(clf.threshold)
clf.threshold = 0.8                       # looks like a plain assignment
print(clf.predict([0.3, 0.75, 0.9]))

try:
    clf.threshold = 50
except ValueError as e:
    print("error:", e)

0.5
[0, 0, 1]
error: threshold must be between 0 and 1, got 50


The real value is stored in `self._threshold` (note the underscore: internal). From outside, `clf.threshold` looks like a normal attribute, but every assignment goes through the check, including the one in `__init__`.

Properties are also good for values that are calculated from other attributes, so they can never be out of date:

In [17]:
class Batch:
    def __init__(self, rows):
        self.rows = rows

    @property
    def shape(self):
        n_cols = len(self.rows[0]) if self.rows else 0
        return (len(self.rows), n_cols)

b = Batch([[1, 2, 3], [4, 5, 6]])
print(b.shape)
b.rows.append([7, 8, 9])
print(b.shape)          # always current, nothing to update by hand

(2, 3)
(3, 3)


Don't overuse them. If something is expensive to compute, a property hides that cost, and a normal method call like `obj.compute_stats()` is more honest.

## 6. Inheritance and abstract base classes

Inheritance lets a class build on another one. The new class (the *subclass*) gets every method of the parent (the *base class*) and can add new ones or replace existing ones.

The main reason ML libraries use inheritance is to define a **common interface**. Every scikit-learn model has `fit`, `predict` and `score`. Because they all share those methods, you can swap one model for another without changing the rest of your code, and tools like cross-validation can work with any model at all.

Let's build that. The base class declares what every model must have, and provides `score` for free:

In [18]:
from abc import ABC, abstractmethod
from collections import Counter


class BaseClassifier(ABC):
    @abstractmethod
    def fit(self, X, y):
        """Learn from features X and labels y. Must return self."""

    @abstractmethod
    def predict(self, X):
        """Return a list of predicted labels."""

    def score(self, X, y):
        predictions = self.predict(X)
        return sum(p == t for p, t in zip(predictions, y)) / len(y)

`ABC` stands for abstract base class. Methods marked `@abstractmethod` have no real body, and they say: "every subclass must implement this". Python enforces it. You can't create an instance of a class that still has abstract methods:

In [19]:
try:
    BaseClassifier()
except TypeError as e:
    print("error:", e)

error: Can't instantiate abstract class BaseClassifier without an implementation for abstract methods 'fit', 'predict'


Now two real classifiers. The first is a **baseline** that ignores the features completely and always predicts the most common label. It sounds useless, but it's one of the most important models you'll ever train. If your fancy model can't beat it, your fancy model hasn't learned anything.

In [20]:
class MajorityClassifier(BaseClassifier):
    def fit(self, X, y):
        self.label_ = Counter(y).most_common(1)[0][0]
        return self

    def predict(self, X):
        return [self.label_ for _ in X]

`class MajorityClassifier(BaseClassifier)` means "inherit from `BaseClassifier`". We only wrote `fit` and `predict`. `score` comes from the parent.

The second classifier learns a single cut-off on one feature: everything above it is class 1. To find the cut-off, it simply tries every value seen in training and keeps the one with the best accuracy.

In [21]:
class StumpClassifier(BaseClassifier):
    """Predict 1 if the feature is >= a learned cut-off, else 0."""

    def __init__(self, feature=0):
        self.feature = feature

    def fit(self, X, y):
        values = sorted({row[self.feature] for row in X})
        best_acc = -1.0
        for cut in values:
            preds = [1 if row[self.feature] >= cut else 0 for row in X]
            acc = sum(p == t for p, t in zip(preds, y)) / len(y)
            if acc > best_acc:
                best_acc, self.cut_ = acc, cut
        return self

    def predict(self, X):
        return [1 if row[self.feature] >= self.cut_ else 0 for row in X]

    def __repr__(self):
        return f"StumpClassifier(feature={self.feature})"

Some data to try them on: hours studied and hours slept, and whether the student passed.

In [22]:
X_train = [[1, 8], [2, 5], [3, 7], [4, 6], [5, 4], [6, 8], [7, 5], [8, 7], [2, 9], [6, 3]]
y_train = [0, 0, 0, 1, 1, 1, 1, 1, 0, 1]
X_test = [[1, 6], [3, 4], [5, 7], [7, 8], [4, 9]]
y_test = [0, 0, 1, 1, 1]

models = [MajorityClassifier(), StumpClassifier(feature=0), StumpClassifier(feature=1)]

for model in models:
    model.fit(X_train, y_train)
    print(f"{model!r:32} test accuracy: {model.score(X_test, y_test):.0%}")

<__main__.MajorityClassifier object at 0x7f39e83f2900> test accuracy: 60%
StumpClassifier(feature=0)       test accuracy: 100%
StumpClassifier(feature=1)       test accuracy: 60%


This loop is the payoff. It doesn't know or care which kind of model it's holding. It calls `fit` and `score`, and each model does the right thing. This is called **polymorphism**, and it's why you can swap a `RandomForestClassifier` for a `LogisticRegression` in scikit-learn by changing one line.

Notice that `MajorityClassifier` printed with the default, ugly repr, because only `StumpClassifier` defined `__repr__`. If we put `__repr__` in the base class, every model would get it automatically.

### super()

When a subclass defines its own `__init__`, the parent's `__init__` doesn't run unless you call it. `super()` gives you access to the parent class:

In [23]:
class Layer:
    def __init__(self, name):
        self.name = name
        self.training = True

    def __repr__(self):
        return f"{type(self).__name__}(name={self.name!r})"


class Dropout(Layer):
    def __init__(self, name, p=0.5):
        super().__init__(name)      # sets name and training
        self.p = p

    def __repr__(self):
        return f"{super().__repr__()[:-1]}, p={self.p})"


d = Dropout("drop1", p=0.2)
print(d, d.training)
print(isinstance(d, Dropout), isinstance(d, Layer))

Dropout(name='drop1', p=0.2) True
True True


If you forget the `super().__init__(name)` call, `d.training` won't exist, and you'll get an `AttributeError` somewhere far away from the actual mistake. In PyTorch, forgetting `super().__init__()` at the top of your model's `__init__` is one of the most common beginner errors, and it produces exactly this kind of confusing message.

`type(self).__name__` in the base class's `__repr__` gives the name of the *actual* class, so subclasses print correctly without writing their own.

### Don't go too deep

Inheritance hierarchies more than two or three levels deep get hard to follow fast, because to understand one class you have to read all its ancestors. Use inheritance for "is a kind of" relationships with a shared interface, and use composition (next section) for everything else.

## 7. Composition

Composition means building an object out of other objects: it *has* them, rather than *being* one of them. It's usually more flexible than inheritance.

Here's an ensemble that holds several classifiers and lets them vote. It doesn't care what kind of classifiers they are, as long as they have `fit` and `predict`.

In [24]:
class VotingClassifier(BaseClassifier):
    def __init__(self, models):
        self.models = models

    def fit(self, X, y):
        for model in self.models:
            model.fit(X, y)
        return self

    def predict(self, X):
        all_preds = [model.predict(X) for model in self.models]
        # all_preds[m][i] is model m's prediction for example i
        return [Counter(column).most_common(1)[0][0] for column in zip(*all_preds)]


ensemble = VotingClassifier([StumpClassifier(0), StumpClassifier(1), MajorityClassifier()])
ensemble.fit(X_train, y_train)
print(f"ensemble test accuracy: {ensemble.score(X_test, y_test):.0%}")

ensemble test accuracy: 60%


The ensemble scored worse than the best single stump. That's not a bug, and it's worth understanding. Voting only helps when the members are each reasonably good *and* make different mistakes, so the majority can outvote any one of them when it's wrong. Here, two of the three members are weak (the majority baseline and the stump on hours slept), so they outvote the one good model. Ensembles are powerful, but "combine more models" doesn't automatically mean "better".

`VotingClassifier` is a classifier (it inherits from `BaseClassifier`, so it gets `score`), and it *has* a list of classifiers. That combination is very common. And because the ensemble follows the same interface as the models inside it, you could even put an ensemble inside another ensemble.

`zip(*all_preds)` is worth a second look. `all_preds` is a list of rows, one row per model. The `*` unpacks it, so `zip` receives each row as a separate argument and pairs them up position by position. The result is one column per example, containing every model's vote for that example. It's the transpose trick from Module 1.

PyTorch models are built almost entirely by composition. A model *has* layers, a layer *has* weights, and calling the model calls each layer in turn.

## 8. Dataclasses

A lot of classes exist mainly to hold data: configs, results, records. Writing `__init__`, `__repr__` and `__eq__` by hand for these is repetitive. The `@dataclass` decorator writes them for you, based on the fields you declare with type hints:

In [25]:
from dataclasses import dataclass, field, asdict


@dataclass
class TrainingConfig:
    model: str = "mlp"
    lr: float = 1e-3
    batch_size: int = 32
    epochs: int = 10
    hidden_sizes: list = field(default_factory=lambda: [64, 32])


config = TrainingConfig(lr=3e-4, epochs=20)
print(config)
print(config.lr)
print(config == TrainingConfig(lr=3e-4, epochs=20))
print(asdict(config))

TrainingConfig(model='mlp', lr=0.0003, batch_size=32, epochs=20, hidden_sizes=[64, 32])
0.0003
True
{'model': 'mlp', 'lr': 0.0003, 'batch_size': 32, 'epochs': 20, 'hidden_sizes': [64, 32]}


From six lines, we got an `__init__` with keyword arguments and defaults, a readable `__repr__`, value-based `__eq__`, and `asdict` for saving the config as JSON.

Look at `hidden_sizes`. You can't write `hidden_sizes: list = [64, 32]`. It's the mutable default trap again, and dataclasses refuse it outright with an error. `field(default_factory=...)` passes a function that gets called to make a fresh list for each new instance.

### Validation with __post_init__

The generated `__init__` doesn't check anything. To validate, add `__post_init__`, which runs right after it:

In [26]:
@dataclass
class TrainingConfig:
    model: str = "mlp"
    lr: float = 1e-3
    batch_size: int = 32
    epochs: int = 10
    hidden_sizes: list = field(default_factory=lambda: [64, 32])

    def __post_init__(self):
        if self.lr <= 0:
            raise ValueError(f"lr must be positive, got {self.lr}")
        if self.batch_size < 1:
            raise ValueError(f"batch_size must be at least 1, got {self.batch_size}")


try:
    TrainingConfig(lr=-0.01)
except ValueError as e:
    print("error:", e)

error: lr must be positive, got -0.01


A typo in a config should fail in the first second, not after an hour of training with a broken setting.

Note that type hints like `lr: float` are not enforced. `TrainingConfig(lr="fast")` would be accepted without complaint. They're documentation, and tools like editors and type checkers use them, but Python itself doesn't check them at runtime. Module 4 covers type hints properly.

### Frozen dataclasses

`@dataclass(frozen=True)` makes instances read-only. That's a good fit for results you don't want anything to change by accident, and frozen instances can be dictionary keys or set members, since they can't change.

In [27]:
@dataclass(frozen=True)
class RunResult:
    run_id: str
    accuracy: float
    epochs: int


result = RunResult("run_042", 0.913, 20)
try:
    result.accuracy = 0.99
except Exception as e:
    print(type(e).__name__, e)

results = [RunResult("a", 0.88, 10), RunResult("b", 0.91, 20), RunResult("c", 0.85, 5)]
best = max(results, key=lambda r: r.accuracy)
print(best)

FrozenInstanceError cannot assign to field 'accuracy'
RunResult(run_id='b', accuracy=0.91, epochs=20)


**When to use what:** use a dataclass when the class is mostly data. Use a regular class when it's mostly behaviour (like a model with `fit` and `predict`). A dictionary is still fine for quick, throwaway structures, but for a config that gets passed around a whole project, a dataclass catches typos (`config.lerning_rate` raises an error, `config["lerning_rate"]` on a dict silently doesn't exist until you read it) and documents what fields exist.

## 9. Class methods and static methods

A normal method receives the instance (`self`). Two other kinds exist.

A **class method** receives the class itself (`cls`) instead. Its main use is alternative constructors: other ways to create an instance. You've probably seen `from_pretrained` in Hugging Face, as in `AutoModel.from_pretrained("bert-base-uncased")`. That's a class method.

In [28]:
@dataclass
class TrainingConfig:
    model: str = "mlp"
    lr: float = 1e-3
    batch_size: int = 32
    epochs: int = 10

    @classmethod
    def from_dict(cls, d):
        known = {k: v for k, v in d.items() if k in cls.__dataclass_fields__}
        unknown = set(d) - set(known)
        if unknown:
            raise ValueError(f"unknown config keys: {sorted(unknown)}")
        return cls(**known)

    @classmethod
    def debug(cls):
        """A tiny config for quickly checking that the code runs end to end."""
        return cls(batch_size=2, epochs=1)


print(TrainingConfig.from_dict({"lr": 0.01, "epochs": 5}))
print(TrainingConfig.debug())

try:
    TrainingConfig.from_dict({"lr": 0.01, "epoch": 5})   # typo
except ValueError as e:
    print("error:", e)

TrainingConfig(model='mlp', lr=0.01, batch_size=32, epochs=5)
TrainingConfig(model='mlp', lr=0.001, batch_size=2, epochs=1)
error: unknown config keys: ['epoch']


Using `cls(...)` instead of `TrainingConfig(...)` inside the method means subclasses automatically get the right type back.

A **static method** receives neither the instance nor the class. It's just a plain function that lives inside a class because it belongs with it logically:

In [29]:
class Tokenizer:
    def __init__(self, lowercase=True):
        self.lowercase = lowercase

    @staticmethod
    def strip_punctuation(text):
        return "".join(ch for ch in text if ch.isalnum() or ch.isspace())

    def __call__(self, text):
        text = self.strip_punctuation(text)
        return (text.lower() if self.lowercase else text).split()


tok = Tokenizer()
print(tok("Hello, World! This is GREAT."))
print(Tokenizer.strip_punctuation("no-self needed!"))

['hello', 'world', 'this', 'is', 'great']
noself needed


Static methods are used less often. If a function doesn't need anything from the class, it can just as well be a normal function at module level.

## 10. Putting it together: Dataset and DataLoader from scratch

PyTorch splits data loading into two classes:

- A **`Dataset`** knows how many examples there are and how to get example number `i`. That's `__len__` and `__getitem__`, which we covered in section 4.
- A **`DataLoader`** takes a dataset and serves it in batches, optionally shuffled, ready for training.

Splitting them up means each class does one job. The same `DataLoader` works with any dataset, whether it reads images from disk, tokenizes text, or generates synthetic data.

Let's build both. First, a base class that defines the interface, and a concrete dataset for tabular data:

In [30]:
class Dataset(ABC):
    @abstractmethod
    def __len__(self):
        ...

    @abstractmethod
    def __getitem__(self, i):
        ...


class TabularDataset(Dataset):
    def __init__(self, X, y):
        if len(X) != len(y):
            raise ValueError(f"X has {len(X)} rows but y has {len(y)}")
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, i):
        return self.X[i], self.y[i]

    def __repr__(self):
        return f"TabularDataset(n={len(self)})"


data = TabularDataset([[i, i * 10] for i in range(10)], [i % 2 for i in range(10)])
print(data, data[3])

TabularDataset(n=10) ([3, 30], 1)


Now the `DataLoader`. Its `__iter__` method is what makes `for batch in loader:` work. It uses `yield`, which hands out one batch, pauses until the `for` loop asks for the next one, then continues from where it left off. That's a *generator*, and Module 4 explains exactly how it works. For now, read `yield batch` as "give this batch to the loop".

In [31]:
import random


class DataLoader:
    def __init__(self, dataset, batch_size=1, shuffle=False, drop_last=False, seed=None):
        if batch_size < 1:
            raise ValueError("batch_size must be at least 1")
        self.dataset = dataset
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.drop_last = drop_last
        self._rng = random.Random(seed)

    def __len__(self):
        """Number of batches per epoch."""
        n = len(self.dataset)
        if self.drop_last:
            return n // self.batch_size
        return (n + self.batch_size - 1) // self.batch_size   # round up

    def __iter__(self):
        indices = list(range(len(self.dataset)))
        if self.shuffle:
            self._rng.shuffle(indices)

        for start in range(0, len(indices), self.batch_size):
            batch_idx = indices[start:start + self.batch_size]
            if self.drop_last and len(batch_idx) < self.batch_size:
                return
            examples = [self.dataset[i] for i in batch_idx]
            xs, ys = zip(*examples)          # [(x, y), ...] -> (x, x, ...), (y, y, ...)
            yield list(xs), list(ys)

A few design points:

- The loader shuffles a list of **indexes**, not the data itself. The dataset is never copied or modified, and it only loads the examples that are actually needed.
- `zip(*examples)` turns a list of `(x, y)` pairs into one tuple of all the `x`s and one of all the `y`s. PyTorch calls this step *collating*.
- `(n + batch_size - 1) // batch_size` is the standard integer trick for dividing and rounding up.
- The loader keeps its own `random.Random`, so a seed makes the shuffling reproducible without affecting anything else in the program.

Let's use it:

In [32]:
loader = DataLoader(data, batch_size=4, shuffle=True, seed=0)
print("batches per epoch:", len(loader))

for epoch in range(2):
    print(f"epoch {epoch}")
    for xb, yb in loader:
        print("  x:", xb, " y:", yb)

batches per epoch: 3
epoch 0
  x: [[7, 70], [8, 80], [1, 10], [5, 50]]  y: [1, 0, 1, 1]
  x: [[3, 30], [4, 40], [2, 20], [0, 0]]  y: [1, 0, 0, 0]
  x: [[9, 90], [6, 60]]  y: [1, 0]
epoch 1
  x: [[9, 90], [4, 40], [8, 80], [6, 60]]  y: [1, 0, 0, 0]
  x: [[0, 0], [1, 10], [7, 70], [2, 20]]  y: [0, 1, 1, 0]
  x: [[3, 30], [5, 50]]  y: [1, 1]


Each epoch gets a different order, which is what you want: if a model sees the examples in the same order every time, it can pick up patterns from the order itself. The last batch is smaller, because 10 isn't divisible by 4. With `drop_last=True` it would be skipped:

In [33]:
print(len(DataLoader(data, batch_size=4, drop_last=True)))
for xb, yb in DataLoader(data, batch_size=4, drop_last=True):
    print(yb)

2
[0, 1, 0, 1]
[0, 1, 0, 1]


And here's how it all fits into a training loop. This is the exact shape of a PyTorch training loop. Only the model is a placeholder.

In [34]:
class MeanModel:
    """Placeholder model: predicts the running mean of the labels it has seen."""

    def __init__(self):
        self.total = 0.0
        self.count = 0

    def __call__(self, xb):
        mean = self.total / self.count if self.count else 0.0
        return [mean for _ in xb]

    def update(self, yb):
        self.total += sum(yb)
        self.count += len(yb)


config = TrainingConfig(batch_size=4, epochs=3)
model = MeanModel()
loader = DataLoader(data, batch_size=config.batch_size, shuffle=True, seed=42)

for epoch in range(config.epochs):
    total_loss = 0.0
    for xb, yb in loader:
        preds = model(xb)
        loss = sum((p - t) ** 2 for p, t in zip(preds, yb)) / len(yb)
        model.update(yb)
        total_loss += loss
    print(f"epoch {epoch}: mean batch loss = {total_loss / len(loader):.4f}")

epoch 0: mean batch loss = 0.3333
epoch 1: mean batch loss = 0.2500
epoch 2: mean batch loss = 0.2500


Config, dataset, loader, model, loop. Every piece is a small class with one job, and they connect through simple interfaces: `len()`, `[]`, `for` and `()`. That's what good object-oriented ML code looks like, and it's exactly how PyTorch is organised.

## Summary

- Use a class when there's **state** to keep between calls: learned parameters, settings, accumulated counts. Otherwise a function is fine.
- `__init__` sets up the instance. `self` is the instance, passed in automatically.
- Follow scikit-learn's conventions: `fit` returns `self`, and learned attributes end with `_`.
- Mutable attributes go in `__init__`, never in the class body.
- Dunder methods plug your objects into Python's syntax: `__repr__`, `__len__`, `__getitem__`, `__iter__`, `__eq__`, `__add__`/`__radd__` and `__call__` are the important ones.
- `@property` gives you validation and always-current computed values with attribute syntax.
- Use inheritance and `ABC` to define a shared interface. Call `super().__init__()` in subclasses.
- Prefer composition, where an object holds other objects, for everything that isn't a clear "is a kind of" relationship.
- `@dataclass` for classes that mostly hold data. Use `field(default_factory=...)` for mutable defaults and `__post_init__` for validation.
- `@classmethod` for alternative constructors like `from_dict` and `from_pretrained`.
- A `Dataset` needs `__len__` and `__getitem__`. A `DataLoader` shuffles indexes and yields batches.

Next, open `02_exercises.ipynb`.